# Text Mining Group Project
## Market Sentiment Based on Tweets
**Master in Data Science & Advanced Analytics — Text Mining Course**
---
### Group Members

Group 27

```
Student 1: Duarte Neves 20221880 
Student 2: Margarida Ourives 20221809
Student 3: Ricardo Trindade 20222039
Student 4: Rodrigo Amaral 20221943
```

```


# Final Full Pipeline: RoBERTa Fine-tuned

In [ ]:
#Imports 
import pandas as pd
import numpy as np
import re
import string
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, f1_score
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer
from nltk import pos_tag

nltk.download('stopwords',                      quiet=True)
nltk.download('wordnet',                        quiet=True)
nltk.download('omw-1.4',                        quiet=True)
nltk.download('averaged_perceptron_tagger',     quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

RANDOM_STATE = 42
BATCH_SIZE   = 32
MAX_LEN      = 128
LR           = 2e-5
EPOCHS       = 13
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

#2. Load Raw Data
train_raw = pd.read_csv('/kaggle/input/datasets/duartecneves/mining-text/train.csv')
test      = pd.read_csv('/kaggle/input/datasets/duartecneves/mining-text/test.csv')
print(f'Train: {len(train_raw)} | Test: {len(test)}')

#3. Preprocessing
lemmatizer = WordNetLemmatizer()
tweet_tok  = TweetTokenizer(strip_handles=False, reduce_len=True)
stop_words = set(stopwords.words('english')) | {'marketscreener', 'rt', 'amp', 'via'}

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    if tag.startswith('V'): return wordnet.VERB
    if tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

def preprocess(text: str) -> str:
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'^RT[\s]+', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'\$([A-Z]{1,5})', r'TICKER_\1', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = tweet_tok.tokenize(text.lower())
    cleaned = []
    for t in tokens:
        if t.startswith('ticker_'):
            cleaned.append(t)
        elif (t not in stop_words
              and t not in string.punctuation
              and not re.match(r'^[^a-z]+$', t)
              and len(t) > 1):
            cleaned.append(t)
    tagged = pos_tag(cleaned)
    tokens = [
        t if t.startswith('ticker_')
        else lemmatizer.lemmatize(t, get_wordnet_pos(pos))
        for t, pos in tagged
    ]
    result = ' '.join(tokens)
    result = re.sub(r'ticker_([a-z]{1,5})', r'$\1', result)
    return result

def preprocess_transformer(text: str) -> str:
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'^RT[\s]+', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply
train_raw['text_clean']       = train_raw['text'].apply(preprocess)
train_raw['text_transformer'] = train_raw['text'].apply(preprocess_transformer)
test['text_clean']            = test['text'].apply(preprocess)
test['text_transformer']      = test['text'].apply(preprocess_transformer)

# Handle empty tweets
train_raw = train_raw[train_raw['text_clean'].str.strip() != ''].reset_index(drop=True)
test.loc[test['text_clean'].str.strip() == '',        'text_clean']        = 'unk'
test.loc[test['text_transformer'].str.strip() == '', 'text_transformer']   = 'unk'

train_clean = train_raw[['text', 'label', 'text_clean', 'text_transformer']]
test        = test[['id', 'text', 'text_clean', 'text_transformer']]
print('Preprocessing done.')

# 4. Corpus Split
train_df, val_df = train_test_split(
    train_clean,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=train_clean['label']
)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test)}')

# 5. Class Weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label']
)
class_weight_dict = dict(enumerate(class_weights))
print('Class weights:', class_weight_dict)

weights_tensor = torch.tensor(
    [class_weight_dict[0], class_weight_dict[1], class_weight_dict[2]],
    dtype=torch.float32
).to(DEVICE)

# 6. Dataset
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

class TweetDatasetNoLabels(Dataset):
    def __init__(self, texts, tokenizer):
        self.encodings = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx]
        }

# 7. Texts & Labels
train_texts  = train_df['text_transformer'].tolist()
val_texts    = val_df['text_transformer'].tolist()
test_texts   = test['text_transformer'].tolist()
train_labels = train_df['label'].tolist()
val_labels   = val_df['label'].tolist()

# 8. Tokenizer & Model
MODEL_NAME = 'roberta-base'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(
                 MODEL_NAME, num_labels=3)
model      = model.to(DEVICE)

# 9. Datasets & Loaders
train_dataset = TweetDataset(train_texts, train_labels, tokenizer)
val_dataset   = TweetDataset(val_texts,   val_labels,   tokenizer)
test_dataset  = TweetDatasetNoLabels(test_texts, tokenizer)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

# 10. Optimizer, Scheduler, Loss
optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler    = get_linear_schedule_with_warmup(
                   optimizer,
                   num_warmup_steps=warmup_steps,
                   num_training_steps=total_steps
               )
loss_fn = torch.nn.CrossEntropyLoss(weight=weights_tensor)

# 11. Training
best_macro_f1    = -1
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs        = model(input_ids=input_ids,
                                   attention_mask=attention_mask)
            all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(batch['labels'].numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    acc      = accuracy_score(all_labels, all_preds)
    print(f'Epoch {epoch+1}/{EPOCHS} — loss: {avg_loss:.4f} '
          f'| val acc: {acc:.4f} | val macro-F1: {macro_f1:.4f}')

    if macro_f1 > best_macro_f1:
        best_macro_f1    = macro_f1
        best_model_state = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
        print(f'  ✓ New best model saved (macro-F1={best_macro_f1:.4f})')

# 12. Load Best Checkpoint & Validate
model.load_state_dict(best_model_state)
model.eval()
all_preds = []
with torch.no_grad():
    for batch in DataLoader(val_dataset, batch_size=BATCH_SIZE):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs        = model(input_ids=input_ids,
                               attention_mask=attention_mask)
        all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())

print(f'\n--- Final Validation Results ---')
print(f'Accuracy: {accuracy_score(val_labels, all_preds):.4f}\n')
print(classification_report(val_labels, all_preds,
                              target_names=['Bearish', 'Bullish', 'Neutral']))

# 13. Test Predictions
test_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs        = model(input_ids=input_ids,
                               attention_mask=attention_mask)
        test_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())

# 14. Save Predictions
pred_df = pd.DataFrame({
    'id':    test['id'],
    'label': test_preds
})
pred_df.to_csv('pred_xx.csv', index=False)
print(f'\nPredictions saved: {pred_df.shape}')
print(pred_df['label'].value_counts().sort_index())

Using device: cuda
Train: 9543 | Test: 2388
Preprocessing done.
Train: 7628 | Val: 1907 | Test: 2388
Class weights: {0: np.float64(2.2052616363110724), 1: np.float64(1.6532293021239706), 2: np.float64(0.5150226183242185)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/13 — loss: 0.8300 | val acc: 0.7897 | val macro-F1: 0.7653
  ✓ New best model saved (macro-F1=0.7653)
Epoch 2/13 — loss: 0.4209 | val acc: 0.8306 | val macro-F1: 0.8049
  ✓ New best model saved (macro-F1=0.8049)
Epoch 3/13 — loss: 0.2767 | val acc: 0.8815 | val macro-F1: 0.8540
  ✓ New best model saved (macro-F1=0.8540)
Epoch 4/13 — loss: 0.1826 | val acc: 0.8673 | val macro-F1: 0.8431
Epoch 5/13 — loss: 0.1289 | val acc: 0.8946 | val macro-F1: 0.8700
  ✓ New best model saved (macro-F1=0.8700)
Epoch 6/13 — loss: 0.1043 | val acc: 0.8878 | val macro-F1: 0.8619
Epoch 7/13 — loss: 0.0814 | val acc: 0.8936 | val macro-F1: 0.8656
Epoch 8/13 — loss: 0.0590 | val acc: 0.8825 | val macro-F1: 0.8547
Epoch 9/13 — loss: 0.0483 | val acc: 0.8799 | val macro-F1: 0.8522
Epoch 10/13 — loss: 0.0242 | val acc: 0.8820 | val macro-F1: 0.8547
Epoch 11/13 — loss: 0.0332 | val acc: 0.8983 | val macro-F1: 0.8709
  ✓ New best model saved (macro-F1=0.8709)
Epoch 12/13 — loss: 0.0176 | val acc: 0.8956 |